In [22]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [23]:
data = load_breast_cancer()

X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [24]:
client_features = [
    list(range(0,8)),      # Client 1 uses features 0-7
    list(range(8,16)),     # Client 2 uses features 8-15
    list(range(16,24)),    # Client 3 uses features 16-23
    list(range(24,30))     # Client 4 uses features 24-29
]

In [25]:
clients = []

for features in client_features:
    
    X_client = torch.zeros_like(X_train)

    X_client[:, features] = X_train[:, features]

    clients.append((X_client, y_train))

In [26]:
class NeuralNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(30,16)
        self.fc2 = nn.Linear(16,8)
        self.fc3 = nn.Linear(8,1)

        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self,x):

        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.sigmoid(self.fc3(x))

        return x

In [27]:
def train_local(model, X, y, epochs=5):

    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    for _ in range(epochs):

        optimizer.zero_grad()

        output = model(X).squeeze()

        loss = criterion(output, y)

        loss.backward()
        optimizer.step()

    return model.state_dict()

In [28]:
def federated_avg(weights):

    avg = {}

    for key in weights[0].keys():
        avg[key] = torch.stack([w[key] for w in weights]).mean(0)

    return avg

In [29]:
global_model = NeuralNet()

rounds = 5

for r in range(rounds):

    print("Round", r+1)

    local_weights = []

    for X_client, y_client in clients:

        local_model = NeuralNet()

        local_model.load_state_dict(global_model.state_dict())

        w = train_local(local_model, X_client, y_client)

        local_weights.append(w)

    new_weights = federated_avg(local_weights)

    global_model.load_state_dict(new_weights)

Round 1
Round 2
Round 3
Round 4
Round 5


In [30]:
global_model = NeuralNet()

rounds = 5

for r in range(rounds):

    print("Round", r+1)

    local_weights = []

    for X_client, y_client in clients:

        local_model = NeuralNet()

        local_model.load_state_dict(global_model.state_dict())

        w = train_local(local_model, X_client, y_client)

        local_weights.append(w)

    new_weights = federated_avg(local_weights)

    global_model.load_state_dict(new_weights)

Round 1
Round 2
Round 3
Round 4
Round 5


In [31]:
with torch.no_grad():

    outputs = global_model(X_test).squeeze()

    preds = (outputs > 0.5).float()

    acc = (preds == y_test).sum().item() / len(y_test)

print("Global Model Accuracy:", acc)

Global Model Accuracy: 0.9736842105263158
